# Generate amorphous LSU networks

This notebook demonstrates `generate_lsu_network`, which implements the
Wooten-Winer-Weaire simulated annealing algorithm of Sellers et al.
(*Nat. Commun.* **8**, 14439, 2017) on a periodic 3-regular graph.

Documentation: see the `claude_context/` folder.

**Inputs**: target LSU (`lsu_degree_12` or `lsu_degree_22`), `num_rods`, `bounds_microns`.

**Output**: NumPy array `(num_rods, 6)` where each row is `[x1, y1, z1, x2, y2, z2]` —
directly usable in the `create_permittivity_grid_penlike` pipeline.

In [1]:
import numpy as np
import os
import lsu_network as lsu
import jax 

print('JAX available:', lsu.HAS_JAX)
print("Devices:", jax.devices())

JAX available: True
Devices: [CudaDevice(id=0)]


## Quick test (24 rods, ~10 s)

Sanity check: small 16-vertex network, target Φ_12 = 0.95.
Useful for verifying the pipeline before launching a long run.

In [2]:
# rods_small = lsu.generate_lsu_network(
#     lsu_degree_12=0.95,
#     num_rods=24,
#     bounds_microns=4.0,
#     edge_length=0.8,
#     n_www_iterations=200,
#     initial_temperature=0.5,
#     final_temperature=1e-3,
#     check_lsu_every=50,
#     relax_global_every=100,
#     seed=1,
#     use_jax=False,  # JIT overhead not worth it at this size
#     verbose=True,
# )
# print('shape:', rods_small.shape, 'dtype:', rods_small.dtype)
# rods_small[:3]

## Reproduce the example: 1653 rods, periodicity 11.44 µm

The reference example has Φ_12 ≈ 0.99 and Φ_22 ≈ 0.89, ~1102 vertices,
1653 rods, periodicity 11.44 µm, mean rod length 0.8 µm.

At full scale this needs ~50,000 WWW iterations. With JAX it's tractable
(JIT-compiled energy + autodiff gradient); without JAX, plan to run
overnight or reduce iterations.

In [3]:
# Adjust n_www_iterations down if you want a quicker (less converged) run.
rods = lsu.generate_lsu_network(
    lsu_degree_22=0.89,            # could also use lsu_degree_12=0.99
    num_rods=1653,
    bounds_microns=11.44,
    edge_length=0.8,
    n_www_iterations=100_000,
    initial_temperature=0.5,
    final_temperature=1e-3,
    target_tolerance=0.01,
    check_lsu_every=100,
    relax_global_every=500,
    relax_local_iters=100,
    relax_global_iters=500,
    seed=42,
    use_jax=True,
    verbose=True,
    energy_weights={'alpha':10, 'beta':1, 'gamma':1, 'delta':1}
)
print('shape:', rods.shape)

[gen] N=1102 vertices, E=1653 rods, box=[11.44, 11.44, 11.44], d0=0.8, target phi_22=0.89, jax=on, jaxopt=off
[gen] BM seed: rod length mean=1.000, std=0.338, min=0.561, max=6.166
[gen] initial relax: E=1500
[WWW it=   100] T=0.4969  E=1378  phi_22=0.5034  acc=29.00%  elapsed=13.5s
[WWW it=   200] T=0.4939  E=1303  phi_22=0.5097  acc=34.00%  elapsed=26.4s
[WWW it=   300] T=0.4908  E=1263  phi_22=0.5150  acc=32.33%  elapsed=39.5s
[WWW it=   400] T=0.4878  E=1238  phi_22=0.5145  acc=32.25%  elapsed=52.9s
[WWW it=   500] T=0.4847  E=1208  phi_22=0.5164  acc=31.20%  elapsed=66.4s
[WWW it=   600] T=0.4817  E=1208  phi_22=0.5232  acc=26.00%  elapsed=71.4s
[WWW it=   700] T=0.4787  E=1208  phi_22=0.5177  acc=22.29%  elapsed=76.5s
[WWW it=   800] T=0.4758  E=1208  phi_22=0.5166  acc=19.50%  elapsed=81.5s
[WWW it=   900] T=0.4728  E=1208  phi_22=0.5164  acc=17.33%  elapsed=86.4s
[WWW it=  1000] T=0.4699  E=1208  phi_22=0.5220  acc=15.60%  elapsed=91.2s
[WWW it=  1100] T=0.467  E=1208  phi_22=0.

## Save the output

The 6-column form (x1,y1,z1,x2,y2,z2) is directly compatible with
`np.loadtxt` as used by the rest of the pipeline. The 7-column form below
(with a 1-based index column) matches `Example/lsu_example_ends.txt`.

In [4]:
os.makedirs('./Example', exist_ok=True)

# 6-column compatible with create_permittivity_grid_penlike
np.savetxt('./Example/lsu_generated.txt', rods,
           fmt=' '.join(['%.14g'] * 6), delimiter='\t')

# # 7-column with index, matching Example/lsu_example_ends.txt
# indexed = np.column_stack([np.arange(1, len(rods) + 1), rods])
# np.savetxt('./Example/lsu_generated_indexed.txt', indexed,
#            fmt='%d\t' + '\t'.join(['%.14g'] * 6))

print('saved', rods.shape[0], 'rods')

saved 1653 rods


## Verify the result

Quick checks: connectivity (rods belong to one connected network), edge length
distribution, and final LSU values.

In [5]:
BOX = 11.44

p1 = rods[:, :3]
p2 = rods[:, 3:]
lengths = np.linalg.norm(p2 - p1, axis=1)

# 1) Rod-length distribution. Reference example (1653 rods, BOX=11.44):
#    mean=0.800 std=0.029  q5=0.752 med=0.801 q95=0.846 min=0.667 max=0.884
qs = np.quantile(lengths, [0.0, 0.05, 0.25, 0.5, 0.75, 0.95, 1.0])
print(f'rod count : {len(rods)}')
print(f'lengths   : mean={lengths.mean():.3f} std={lengths.std():.3f}')
print(f'  quartiles  min={qs[0]:.3f}  5%={qs[1]:.3f}  Q1={qs[2]:.3f}  '
      f'med={qs[3]:.3f}  Q3={qs[4]:.3f}  95%={qs[5]:.3f}  max={qs[6]:.3f}')
print(f'  ref target  mean=0.800 std=0.029 (reach with enough WWW iters)')

# 2) Spatial-coverage check — tile the canonical box into 1 µm cells and
# count cells with no vertex. Reference example: 54.5% empty (Poisson at
# density 0.74 verts/µm³ would naturally give ~44% empty). The thing the
# old configuration-model seed got wrong was *clusters* of empty cells —
# multi-µm voids. The Poisson-disk seed used now should give a roughly
# Poisson-like empty-cell pattern with no large connected void region.
half = BOX / 2.0
verts = np.vstack([p1, p2])
verts_canon = verts - BOX * np.round(verts / BOX)
n_cells = int(np.ceil(BOX))
edges_grid = np.linspace(-half, half, n_cells + 1)
H, _ = np.histogramdd(verts_canon, bins=(edges_grid, edges_grid, edges_grid))
empty = int(np.sum(H == 0))
print(f'1 µm³ vertex coverage: {H.size} cells, {empty} empty '
      f'({empty / H.size:.1%})  (reference: 54.5%)')

try:
    from scipy.ndimage import label
    labeled, n_components = label(H == 0)
    sizes = sorted((int((labeled == c).sum()) for c in range(1, n_components + 1)),
                   reverse=True)
    print(f'  largest empty clusters: {sizes[:5]}  '
          f'(big single cluster is normal at this density due to PBC '
          f'percolation; what was *wrong* before was a big cluster on a '
          f'box face)')
except ImportError:
    pass

print(f'box span   : x [{p1[:,0].min():.3f}, {p1[:,0].max():.3f}] '
      f'y [{p1[:,1].min():.3f}, {p1[:,1].max():.3f}] '
      f'z [{p1[:,2].min():.3f}, {p1[:,2].max():.3f}]')

rod count : 1653
lengths   : mean=0.816 std=0.039
  quartiles  min=0.702  5%=0.757  Q1=0.787  med=0.813  Q3=0.841  95%=0.886  max=0.964
  ref target  mean=0.800 std=0.029 (reach with enough WWW iters)
1 µm³ vertex coverage: 1728 cells, 895 empty (51.8%)  (reference: 54.5%)
  largest empty clusters: [852, 6, 4, 2, 2]  (big single cluster is normal at this density due to PBC percolation; what was *wrong* before was a big cluster on a box face)
box span   : x [-5.716, 5.709] y [-5.719, 5.709] z [-5.715, 5.704]
